# 3.3 Magnetization Square by Classical Shadow

---

## Basic Usage


### a. Import the instances


In [1]:
from qurry import ShadowUnveil

experiment_shadow = ShadowUnveil()

### b. Preparing quantum circuit


In [2]:
from qurry.recipe import trivial_paramagnet, ghz


In [3]:
circuits_dict = {
    "trivialPM_2": trivial_paramagnet(2),
    "trivialPM_4": trivial_paramagnet(4),
    "trivialPM_6": trivial_paramagnet(6),
    "trivialPM_8": trivial_paramagnet(8),
    "cat_2": ghz(2),
    "cat_4": ghz(4),
    "cat_6": ghz(6),
    "cat_8": ghz(8),
}
print("| trivial paramagnet and cat in 4 qubits:")
print(circuits_dict["trivialPM_4"])
print(circuits_dict["cat_4"])

| trivial paramagnet and cat in 4 qubits:
     ┌───┐
q_0: ┤ H ├
     ├───┤
q_1: ┤ H ├
     ├───┤
q_2: ┤ H ├
     ├───┤
q_3: ┤ H ├
     └───┘
     ┌───┐               
q_0: ┤ H ├──■────────────
     └───┘┌─┴─┐          
q_1: ─────┤ X ├──■───────
          └───┘┌─┴─┐     
q_2: ──────────┤ X ├──■──
               └───┘┌─┴─┐
q_3: ───────────────┤ X ├
                    └───┘


### c. Execute the circuit


#### i. Preparing the operators for post-processing


In [4]:
import numpy as np
import functools as ft
from itertools import permutations
from qiskit import QuantumCircuit
from qiskit.circuit.library import ZGate, IGate


def operator_preparing(
    circuit: QuantumCircuit,
) -> list[np.ndarray[tuple[int, int], np.dtype[np.complex128]]]:
    """Prepare the operator for the circuit.

    Args:
        circuit (QuantumCircuit): The quantum circuit for which the operator is prepared.

    Returns:
        list[np.ndarray[tuple[int, int], np.dtype[np.complex128]]]:
            A list of numpy arrays representing the operator for each pair of qubits.
    """
    z_gate_matrix = ZGate().to_matrix()
    i_gate_matrix = IGate().to_matrix()
    num_qubits = circuit.num_qubits

    return [
        ft.reduce(
            np.kron,
            (
                z_gate_matrix.copy() if i in tgt else i_gate_matrix.copy()
                for i in range(num_qubits)
            ),
        )
        for tgt in permutations(range(num_qubits), 2)
    ]  # type: ignore[return]

In [5]:
operators_for_magnet_sq = operator_preparing(circuits_dict["cat_4"])
print(
    "| The number of operators for Magnetization Square:", len(operators_for_magnet_sq)
)

| The number of operators for Magnetization Square: 12


#### ii. Find a Proper Number of Classical Snapshot for Epsilon Upperbound


In [6]:
from qurry.process.classical_shadow import worst_accuracy_predict_epsilon_calc

In [7]:
num_classical_snapshot_candinates = []

for i in range(1, 6):
    num_classical_snapshot_candinates.append(i * 100)
    print(
        "| The candidate number for classical snapshot:",
        num_classical_snapshot_candinates[-1],
    )
    epsilon_upperbound, shadow_norm_upperbound = worst_accuracy_predict_epsilon_calc(
        num_classical_snapshot_candinates[-1], operators_for_magnet_sq
    )
    print(
        "| The epsilon upper bound for the worst accuracy prediction is :\n",
        epsilon_upperbound,
    )

| The candidate number for classical snapshot: 100
| The epsilon upper bound for the worst accuracy prediction is :
 9.329523031752482
| The candidate number for classical snapshot: 200
| The epsilon upper bound for the worst accuracy prediction is :
 6.596969000988257
| The candidate number for classical snapshot: 300
| The epsilon upper bound for the worst accuracy prediction is :
 5.386402633793108
| The candidate number for classical snapshot: 400
| The epsilon upper bound for the worst accuracy prediction is :
 4.664761515876241
| The candidate number for classical snapshot: 500
| The epsilon upper bound for the worst accuracy prediction is :
 4.172289539329696


We use `300` as the number of classical snapshot.

#### iii. Execute experiment and run post-processing


In [8]:
exp1 = experiment_shadow.measure(circuits_dict["cat_4"], snapshots=300, shots=32)
exp1

<SUExperiment(
  exp_id='76fdf3bd-c2cc-4e48-9d25-a4d6365c943b',
  args=SUArguments(exp_name='experiment.N_U_300.classical_shadow', snapshots=300, qubits_measured=[0, 1, 2, 3], registers_mapping={0: 0, 1: 1, 2: 2, 3: 3}, actual_num_qubits=4, unitary_located=[0, 1, 2, 3], shadow_basis=ShadowRandomBasis(name='RX_RY_RZ'), random_basis={0: {0: 1, 1: 1, 2: 1, 3: 1}, 1: {0: 0, 1: 1, 2: 0, 3: 0}, 2: {0: 0, 1: 1, 2: 2, 3: 1}, 3: {0: 0, 1: 2, 2: 0, 3: 1}, 4: {0: 1, 1: 1, 2: 2, 3: 2}, 5: {0: 0, 1: 0, 2: 0, 3: 1}, 6: {0: 0, 1: 2, 2: 1, 3: 1}, 7: {0: 0, 1: 2, 2: 1, 3: 1}, 8: {0: 1, 1: 0, 2: 1, 3: 0}, 9: {0: 0, 1: 0, 2: 1, 3: 0}, 10: {0: 2, 1: 1, 2: 1, 3: 0}, 11: {0: 0, 1: 2, 2: 2, 3: 2}, 12: {0: 2, 1: 2, 2: 1, 3: 2}, 13: {0: 2, 1: 0, 2: 0, 3: 1}, 14: {0: 0, 1: 2, 2: 1, 3: 0}, 15: {0: 1, 1: 1, 2: 2, 3: 0}, 16: {0: 1, 1: 2, 2: 0, 3: 1}, 17: {0: 2, 1: 0, 2: 0, 3: 2}, 18: {0: 2, 1: 1, 2: 1, 3: 2}, 19: {0: 2, 1: 1, 2: 0, 3: 2}, 20: {0: 0, 1: 0, 2: 1, 3: 2}, 21: {0: 2, 1: 0, 2: 2, 3: 0}, 22: {0: 1, 1: 0,

Use the operators we prepare for post-processing.


In [9]:
# This will take longer to run
report01 = exp1.analyze(
    range(4),
    operators_for_magnet_sq,
)
report01

<SUAnalysis(serial=0, SUProcessEntries(shots=32, random_basis_array=[...300 items...], selected_classical_registers=[0, 1, 2, 3], given_operators=[...12 items...], accuracy_predict_epsilon=0.01, maximum_shadow_norm=None, rho_method=<RhoMethod.MULTI_SHOTS: 'multi_shots'>, shadow_basis=ShadowRandomBasis(name='RX_RY_RZ'), trace_method=<TraceMethod.NOMATMUL_TRACE_RUST: 'nomatmul_trace_rust'>, estimate_trace_method=<ListTraceMethod.EINSUM_AIJ_BJI_TO_AB_JAX: 'einsum_aij_bji_to_ab_jax'>), unused_args_num=0)

In [10]:
report01.results_fields()

(('basic', 'average_snapshots_rho_list'),
 ('basic', 'classical_registers_actually'),
 ('basic', 'taking_time'),
 ('basic', 'rho_method'),
 ('basic', 'random_basis_data'),
 ('basic', 'mean_of_rho'),
 ('purity', 'purity'),
 ('purity', 'entropy'),
 ('purity', 'purity_value_kind'),
 ('purity', 'taking_time'),
 ('purity', 'trace_method'),
 ('estimation', 'estimate_of_given_operators'),
 ('estimation', 'corresponding_rhos'),
 ('estimation', 'accuracy_prob_comp_delta'),
 ('estimation', 'num_of_estimators_k'),
 ('estimation', 'accuracy_predict_epsilon'),
 ('estimation', 'maximum_shadow_norm'),
 ('estimation', 'epsilon_upperbound'),
 ('estimation', 'shadow_norm_upperbound'),
 ('estimation', 'taking_time'),
 ('estimation', 'estimate_trace_method'))

Now, you can find we have some values in `estimate_of_given_operators`, we will need this to calculate magnetization square.


In [11]:
print("| The estimate_of_given_operators:")
print(report01["estimation", "estimate_of_given_operators"])
# or
print(report01.results["estimation"].estimate_of_given_operators)

| The estimate_of_given_operators:
[np.complex128(1+0j), np.complex128(0.9999999999999998+0j), np.complex128(1+0j), np.complex128(1+0j), np.complex128(0.5+0j), np.complex128(0.9999999999999998+0j), np.complex128(0.9999999999999998+0j), np.complex128(0.5+0j), np.complex128(0.75+0j), np.complex128(1+0j), np.complex128(0.9999999999999998+0j), np.complex128(0.75+0j)]
[np.complex128(1+0j), np.complex128(0.9999999999999998+0j), np.complex128(1+0j), np.complex128(1+0j), np.complex128(0.5+0j), np.complex128(0.9999999999999998+0j), np.complex128(0.9999999999999998+0j), np.complex128(0.5+0j), np.complex128(0.75+0j), np.complex128(1+0j), np.complex128(0.9999999999999998+0j), np.complex128(0.75+0j)]


With following functions:


In [12]:
def unveil_magnetization_square(
    estimate_of_given_operators: list[np.complex128],
    num_qubits: int,
) -> np.float64:
    """Processing Classical Shadows post-processing for MagnetSquare.

    Args:
        estimate_of_given_operators (list[np.complex128]): The estimates of the given operators.
        num_qubits (int): The number of qubits in the circuit.

    Returns:
        np.float64: The unveiled magnet square value.
    """
    return np.complex128(sum(estimate_of_given_operators) + num_qubits).real / (
        num_qubits**2
    )

In [13]:
unveil_magnetization_square_01 = unveil_magnetization_square(
    report01.results["estimation"].estimate_of_given_operators,  # type: ignore[arg-type]
    report01.middleware_entries.num_qubits,
)
print(
    "| The magnetization square value is:",
    unveil_magnetization_square_01,
)

| The magnetization square value is: 0.90625


- All main quantities


In [14]:
print(report01.results["basic"].mean_of_rho.shape)  # type: ignore[attr-defined]

(16, 16)


### d. Export them after all


In [15]:
exp1_id, exp1_files_info = exp1.write(
    save_location=".",  # where to save files
)
exp1_files_info

{'save_location': '.',
 'folder': 'experiment.N_U_300.classical_shadow.001',
 'qurryinfo': 'experiment.N_U_300.classical_shadow.001/qurryinfo.json',
 'args': 'experiment.N_U_300.classical_shadow.001/args/id=76fdf3bd-c2cc-4e48-9d25-a4d6365c943b.args.json',
 'advent': 'experiment.N_U_300.classical_shadow.001/advent/id=76fdf3bd-c2cc-4e48-9d25-a4d6365c943b.advent.json',
 'legacy': 'experiment.N_U_300.classical_shadow.001/legacy/id=76fdf3bd-c2cc-4e48-9d25-a4d6365c943b.legacy.json',
 'tales': 'experiment.N_U_300.classical_shadow.001/tales/id=76fdf3bd-c2cc-4e48-9d25-a4d6365c943b.tales.json',
 'myths': 'experiment.N_U_300.classical_shadow.001/myths/id=76fdf3bd-c2cc-4e48-9d25-a4d6365c943b.myths.json'}

### f. Read the saved results


In [16]:
read_result_new = experiment_shadow.experiment_instance.read(
    "experiment.N_U_300.classical_shadow.001",
    save_location=".",  # where files are saved
)

## Post-Process Availablities and Version Info


In [17]:
from qurry.process import AVAIBILITY_STATESHEET

AVAIBILITY_STATESHEET

 | Qurrium version: 1.0.0.dev2
---------------------------------------------------------------------------
 ### Qurrium Post-Processing
   - Backend Availability ................... Python Cython Rust   JAX   
 - randomized_measure
   - entangled_entropy.entropy_core_2 ....... Yes    No     Yes    No    
   - entangled_entropy_v1.entropy_core ...... Yes    Depr.  Yes    No    
   - wavefunction_overlap.echo_core_2 ....... Yes    No     Yes    No    
   - wavefunction_overlap_v1.echo_core ...... Yes    Depr.  Yes    No    
 - hadamard_test
   - purity_echo_core ....................... Yes    No     Yes    No    
 - magnet_square
   - magnsq_core ............................ Yes    No     Yes    No    
 - string_operator
   - strop_core ............................. Yes    No     Yes    No    
 - classical_shadow
   - rho_process ............................ Yes    No     Yes    No    
   - matrix_calculation ..................... Yes    No     No     Yes   
 - utils
   - randomized ....